<a href="https://colab.research.google.com/github/Samu24042/CienciaDeDatos/blob/main/icd_taller3_va_continuas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **¡¡ ANTES DE EMPEZAR !!**

Deshabilita autocompletar con IA. Esta acción te ayudará a aprender de verdad. Si estás en Colab sigue estos pasos:



1.   Ir a Herramientas \ Configuración \ Asistencia de IA
2.   Desactivar la casilla **"Mostrar autocompletado impulsado por IA"**
3.   Activar la casilla **"Ocultar funciones de IA generativa"**


# **SIMULACIÓN DE VARIABLES ALEATORIAS A PARTIR DE DISTRIBUCIONES**

Antes de trabajar con variables aleatorias continuas, apliquemos el método de la inversa de la CDF para generar variables aleatorias con cierta distribución.

# Reto 0: La Cafetería:

Usted está asesorando a una cafetería local para estimar sus ganancias diarias. Sabe que reciben **150 clientes al día** entre las 8:00 a.m. y las 8:00 p.m. (20:00).

A través de la observación, ha determinado dos comportamientos clave:

**1. La Hora de Llegada:**
Los clientes no llegan de manera uniforme. Hay un pico de oficinistas a las 10:00 a.m. y un pico de estudiantes a las 3:00 p.m. (15:00). La probabilidad de que un cliente llegue en una hora específica está dada por la siguiente Función de Masa:
* `Horas: 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20`
* `Probabilidades: 2%, 8%, 20%, 10%, 5%, 5%, 10%, 20%, 10%, 5%, 3%, 1%, 1%`

**2. El Tipo de Consumo:**
* **El Turno Mañana (Antes de las 12:00 m):** La gente tiene prisa. Tienen un **80%** de probabilidad de pedir el *Café Estándar* (\$1 USD) y un 20% de pedir el *Café de Especialidad* (\$2 USD).
* **El Turno Tarde (12:00 m en adelante):** La gente va a relajarse o estudiar. Tienen un **70%** de probabilidad de pedir el *Café de Especialidad* y un 30% de pedir el *Estándar*.

**Entregable:**
Simular 1,000 días completos de operación para encontrar la distribución de las ganancias diarias. ¿Cuál es la probabilidad de que las ganancias estén por debajo de \$ 250 USD?

**Nota:** La función `np.random.rand()` disponible en la librería `numpy` genera un número aleatorio entre 0 y 1 con distribución uniforme.

In [ ]:
# --- CÓDIGO PROBLEMA CAFETERÍA ---


# **VARIABLES ALEATORIAS CONTINUAS**

Vamos a analizar el **S&P 500**, un índice que agrupa el valor de las acciones de las 500 empresas más grandes de Estados Unidos (Apple, Microsoft, Amazon, etc.). Este índice no solo es el termómetro más preciso de la economía global, sino que es el estándar contra el cual los inversores comparan el rendimiento de sus carteras en bolsa, e incluso de otras alternativas de inversión como los CDTs o el sector inmobiliario

En finanzas, rara vez analizamos el precio bruto de una acción. Lo que realmente nos importa es el **Retorno Diario**, es decir, qué porcentaje subió o bajó el precio respecto al día anterior. El retorno diario es un buen ejemplo de una **Variable Aleatoria Continua**:

**Haremos lo siguiente:**
1. Descargar los datos históricos del S&P 500 de los últimos 5 años.
2. Calcular los retornos diarios porcentuales.
3. Graficar un histograma para descubrir la "forma" de esta variable aleatoria.

In [ ]:
# ==========================================
# 0. PREPARACIÓN E INSTALACIÓN DE LIBRERÍAS
# ==========================================

# Instalamos la librería yfinance (Yahoo Finance)
!pip install yfinance -q

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuramos un estilo limpio para las gráficas
plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
# ==========================================
# 1. DESCARGA DE DATOS
# ==========================================

print("Descargando datos del S&P 500...")
sp500 = yf.download('^GSPC', period='5y')

# Seleccionamos el cierre. Usamos doble corchete [['Close']] para asegurar
# que siga siendo un DataFrame (y no una Serie) y hacemos una copia limpia
sp500 = sp500[['Close']].copy()

# Veamos cómo son los datos que acabamos de descargar:
display(sp500.head())

# 1. Renombramos el título del índice (que originalmente es 'Date')
sp500 = sp500.rename_axis('fecha')

# 2. Renombramos la única columna de datos que tenemos
sp500.columns = ['precio']

print("\nPrimeros registros del precio:")
display(sp500.head())


Para analizar una serie de tiempo financiera, el precio diario importa menos que los retornos porcentuales.
$$
  \text{retorno_diario}=\frac{\text{precio_actual}-\text{precio_ayer}}{\text{precio_ayer}}
$$

Vamos a calcular estos retornos en nuestros datos

In [ ]:
# ==========================================
# 2. PRE-PROCESAMIENTO
# ==========================================

# Usando.shift(1) movemos los datos una posición hacia abajo (el precio de ayer).

sp500['precio_ayer'] = sp500['precio'].shift(1)

# Calculamos el retorno porcentual de forma manual (para entender la matemática)
sp500['retorno_diario'] = (sp500['precio'] - sp500['precio_ayer']) / sp500['precio_ayer']

# Forma optimizada de Pandas (hace lo mismo que arriba)
# sp500['retorno_diario'] = sp500['precio'].pct_change() # Descomenta esta línea para usar la versión optimizada

# Limpiamos los datos nulos generados por el shift (la primera fila no tiene "ayer")
sp500.dropna(inplace=True)

display(sp500.head())


## 1. Modelo de la Incertidumbre Financiera

Ahora que tenemos nuestra variable aleatoria continua (`retorno_diario`), vamos a graficar su histograma.

**Pregunta para analizar al ver la gráfica:** ¿Te recuerda a alguna de las distribuciones teóricas que vimos en las diapositivas de clase?

In [ ]:
# ==========================================
# 3. HISTOGRAMA
# ==========================================

retornos = sp500['retorno_diario']*100 # Multiplicamos por 100 para verlo en porcentaje

plt.figure(figsize=(10, 6))
# Usamos density=True para que el área del histograma sume 1 (como una Función de Densidad PDF)
# Si density=False, en el eje vertical se mostrará el conteo de los datos que caen en cada bin
plt.hist(retornos, bins=80, color='#3b82f6', edgecolor='white', density=True, alpha=0.7)

plt.axvline(0, color='red', linestyle='dashed', linewidth=1.5, label='Retorno Neutro (0%)')

plt.title('Distribución de los Retornos Diarios del S&P 500 (Últimos 5 años)')
plt.xlabel('Retorno Diario (%)')
plt.ylabel('Densidad')
plt.legend()
plt.show()

# CALCULE EL RETORNO ANUAL PROMEDIO. ¿ES RENTABLE INVERTIR EN LA BOLSA?


## 2. Cuartiles y Percentiles

Hasta ahora hemos visto el histograma, que nos muestra dónde se agrupan los datos. Pero, ¿cómo podemos *ranquear* esos datos? Para eso usamos los **Percentiles**.

Imaginemos que tomamos los 1,250 días de retornos del S&P 500 y los formamos en una fila:
* A la extrema izquierda ponemos el peor día de todos (la mayor caída).
* A la extrema derecha ponemos el mejor día (la mayor ganancia).
* En el medio quedan los días "normales".

Si empezamos a caminar desde la izquierda (los peores días) hacia la derecha, y nos detenemos cuando hemos pasado exactamente al **10%** de los datos de la fila, el valor del retorno en ese punto exacto es el **Percentil 10**. Significa que el 10% de los días fueron peores que ese valor, y el 90% fueron mejores.

Matemáticamente, el **Percentil $p$** es el valor $x$ donde la probabilidad acumulada es exactamente $p\%$.
* En fórmula, es el valor de $x$ para el que $F_X(x) = p/100$

Por su parte, los **Cuartiles** son simplemente percentiles con nombres especiales que cortan nuestra fila de datos en cuatro pedazos iguales (de 25% en 25%):
* **Q1 (Primer Cuartil o Percentil 25):** Sólo el 25% de los datos es menor a este valor.
* **Q2 (Mediana o Percentil 50):** Corta la fila exactamente por la mitad.
* **Q3 (Tercer Cuartil o Percentil 75):** El 75% de los datos es menor a este valor.

In [ ]:
# ==========================================
# 4. CÁLCULO DE CUARTILES
# ==========================================


# En Pandas, la función .quantile() hace exactamente el trabajo de
# "ordenar la fila y buscar la posición" por nosotros. Recibe un valor entre 0 y 1.

q1 = retornos.quantile(0.25) # Avanzamos el 25% de la fila
q2 = retornos.quantile(0.50) # Avanzamos a la mitad exacta (Mediana)
q3 = retornos.quantile(0.75) # Avanzamos el 75% de la fila

print("--- LOS CUARTILES DEL S&P 500 ---")
print(f"Q1 (Percentil 25): {q1:.3f}%  -> Solo el 25% de los días el mercado rindió menos que esto.")
print(f"Q2 (Percentil 50): {q2:.3f}%  -> La mitad de los días fueron peores que esto.")
print(f"Q3 (Percentil 75): {q3:.3f}%  -> El 75% de los días fueron peores que esto.")


# Reto 1: Peores Escenarios y los "Cisnes Negros"

En finanzas, un "Cisne Negro" es un evento extremadamente raro y de gran impacto. Vamos a buscar esos días de pánico en el histórico del S&P 500.

**Entregables:**
1. **Calcular el Percentil 1** de los retornos diarios (umbral del 1% de los peores días posibles).
2. Si alguien tiene invertido en el S&P 500 10.000 USD ¿a cuántos dólares de pérdida equivale ese porcentaje?
3. **Caza de Cisnes Negros:** Usando el valor del percentil calculado para filtrar el DataFrame `sp500`, mostrar las fechas exactas donde el mercado tuvo retornos peores que ese umbral.
4. ¿Puedes relacionar alguno de estos peores días con algún evento histórico o crisis mundial reciente?

In [ ]:
# --- CÓDIGO PROBLEMA CISNES NEGROS ---
